# Wiggly circle example



### Setup

Say we have $r=1+a \sin(a \theta), \theta \in [0,2\pi], a \in \mathbb{Z} \setminus 1$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider
from scipy.ndimage import gaussian_filter1d


def compute_exact_density_and_mode(a, sigma, phase=0.0, grid_res=500, num_theta=200, num_rays=500):
    theta = np.linspace(0, 2 * np.pi, num_theta)
    r_true = 1 + (1.0 / a) * np.sin(a * theta + phase)
    dr_dtheta = np.cos(a * theta + phase)

    x_c = r_true * np.cos(theta)
    y_c = r_true * np.sin(theta)

    ds = np.sqrt(r_true**2 + dr_dtheta**2)
    total_arc_length = np.trapezoid(ds, x=theta)

    limit = 2
    x_span = np.linspace(-limit, limit, grid_res)
    y_span = np.linspace(-limit, limit, grid_res)
    X, Y = np.meshgrid(x_span, y_span)

    density_grid = np.zeros_like(X)
    for i in range(grid_res):
        X_row = X[i, :][..., np.newaxis]
        Y_row = Y[i, :][..., np.newaxis]
        sq_dist = (X_row - x_c)**2 + (Y_row - y_c)**2
        integrand = np.exp(-sq_dist / (2 * sigma**2)) * ds
        density_grid[i, :] = np.trapezoid(integrand, x=theta, axis=-1) / (2 * np.pi * sigma**2 * total_arc_length)

    phi = np.linspace(0, 2 * np.pi, num_rays)
    R_span = np.linspace(0, limit * 0.9, grid_res)
    PHI, R_grid = np.meshgrid(phi, R_span)

    X_ray = R_grid * np.cos(PHI)
    Y_ray = R_grid * np.sin(PHI)

    density_ray = np.zeros_like(X_ray)
    for i in range(grid_res):
        X_r_row = X_ray[i, :][..., np.newaxis]
        Y_r_row = Y_ray[i, :][..., np.newaxis]
        sq_dist_ray = (X_r_row - x_c)**2 + (Y_r_row - y_c)**2
        integrand_ray = np.exp(-sq_dist_ray / (2 * sigma**2)) * ds
        density_ray[i, :] = np.trapezoid(integrand_ray, x=theta, axis=-1)

    # Sub-pixel parabolic refinement to avoid argmax quantization artifacts
    idx_c = np.clip(np.argmax(density_ray, axis=0), 1, grid_res - 2)
    j = np.arange(num_rays)
    y0 = density_ray[idx_c - 1, j]
    y1 = density_ray[idx_c,     j]
    y2 = density_ray[idx_c + 1, j]
    denom = y0 - 2 * y1 + y2
    offset = np.where(np.abs(denom) > 1e-30, 0.5 * (y0 - y2) / denom, 0.0)
    dr = R_span[1] - R_span[0]
    R_mode = R_span[idx_c] + offset * dr

    # Smooth in polar space to remove residual numerical noise (wrap for periodicity)
    R_mode = gaussian_filter1d(R_mode, sigma=num_rays / 200, mode='wrap')

    x_mode = R_mode * np.cos(phi)
    y_mode = R_mode * np.sin(phi)

    return X, Y, density_grid, x_c, y_c, x_mode, y_mode


def plot_density(a=8, sigma=0.10, x_coord=0.0, y_coord=0.0):
    X, Y, exact_density, x_true, y_true, x_mode, y_mode = compute_exact_density_and_mode(a, sigma)

    fig, ax = plt.subplots(figsize=(8, 8))
    cm = ax.pcolormesh(X, Y, exact_density, shading='auto', cmap='viridis')
    plt.colorbar(cm, ax=ax, label=r'$p_\sigma(x,y)$')
    ax.plot(x_true, y_true, color='black', linewidth=1.5, linestyle='-', label='True curve $C$')
    ax.plot(np.append(x_mode, x_mode[0]), np.append(y_mode, y_mode[0]),
            color='red', linewidth=2.0, linestyle='-', label='Mode curve $M$')

    ax.scatter(x_coord, y_coord, color='orange', marker='.', s=100, linewidth=2,
               label=f'Point ({x_coord:.1f}, {y_coord:.1f})')

    ax.set_aspect('equal')
    ax.set_title(rf'Exact density — $a={a}$, $\sigma={sigma:.2f}$', fontsize=12)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.legend(loc='upper right')
    plt.show()


interact(
    plot_density,
    a=IntSlider(value=8, min=2, max=16, step=1, description='a'),
    sigma=FloatSlider(value=0.10, min=0.01, max=0.7, step=0.01, description='σ', readout_format='.2f'),
    x_coord=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description='x'),
    y_coord=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description='y'),
)

interactive(children=(IntSlider(value=8, description='a', max=16, min=2), FloatSlider(value=0.1, description='…

<function __main__.plot_density(a=8, sigma=0.1, x_coord=0.0, y_coord=0.0)>

## Recap on mathematics
### System Setup and Definitions

Let $y \in \mathbb{R}^d$ be a random variable distributed according to an arbitrary data distribution $p(y)$. Let $x \in \mathbb{R}^d$ be the corresponding noise-perturbed variable generated via the conditional distribution:

$$p_\sigma(x|y) = \mathcal{N}(x; y, \sigma^2 I_d) = \frac{1}{(2\pi\sigma^2)^{d/2}} \exp\left( -\frac{\|x - y\|^2}{2\sigma^2} \right)$$

The marginal distribution $p_\sigma(x)$ is defined by the convolution:

$$p_\sigma(x) = \int_{\mathbb{R}^d} p(y) p_\sigma(x|y) \, dy$$

By Bayes' theorem, the posterior distribution of the clean data $y$ given the observed noisy vector $x$ is:

$$p_\sigma(y|x) = \frac{p(y) p_\sigma(x|y)}{p_\sigma(x)}$$


### Jacobian of score function

$$\nabla_x^2 \log p_\sigma(x) = \frac{1}{\sigma^4} \operatorname{Cov}_{y \sim p_\sigma(y|x)}(y) - \frac{1}{\sigma^2} I_d$$

where $\operatorname{Cov}_{y \sim p_\sigma(y|x)}(y) \in \mathbb{R}^{d \times d}$ is the conditional covariance matrix of the clean data given the noisy observation:

$$\operatorname{Cov}_{y \sim p_\sigma(y|x)}(y) = \mathbb{E}_{y \sim p_\sigma(y|x)}\left[ y y^T \right] - \mathbb{E}_{y \sim p_\sigma(y|x)}[y] \mathbb{E}_{y \sim p_\sigma(y|x)}[y]^T$$


### First-Order Derivative (The Score Function)
We begin by evaluating the spatial gradient of the log-marginal density:

$$\text{score}(x) = \nabla_x \log p_\sigma(x) = \frac{\nabla_x p_\sigma(x)}{p_\sigma(x)}$$

Differentiating the convolution integral under the integral sign yields:

$$\nabla_x p_\sigma(x) = \int_{\mathbb{R}^d} p(y) \nabla_x p_\sigma(x|y) \, dy$$

The gradient of the Gaussian kernel with respect to $x$ is:

$$\nabla_x p_\sigma(x|y) = p_\sigma(x|y) \left( \frac{y - x}{\sigma^2} \right)$$

Substituting this back into the gradient equation and normalizing by $p_\sigma(x)$ provides:

$$\begin{aligned}
\nabla_x \log p_\sigma(x) &= \int_{\mathbb{R}^d} \frac{p(y) p_\sigma(x|y)}{p_\sigma(x)} \left( \frac{y - x}{\sigma^2} \right) \, dy \\
&= \int_{\mathbb{R}^d} p_\sigma(y|x) \left( \frac{y - x}{\sigma^2} \right) \, dy \\
&= \frac{\mathbb{E}_{y \sim p_\sigma(y|x)}[y] - x}{\sigma^2}
\end{aligned}$$

> **Tweedie's Formula:**
> $$\nabla_x \log p_\sigma(x) = \frac{\mathbb{E}_{y \sim p_\sigma(y|x)}[y] - x}{\sigma^2}$$


In [5]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import VBox, HBox

def compute_local_properties(x0, y0, a, sigma, phase=0.0, num_theta=1000):
    theta = np.linspace(0, 2 * np.pi, num_theta)
    r_true = 1 + (1.0 / a) * np.sin(a * theta + phase)
    x_c = r_true * np.cos(theta)
    y_c = r_true * np.sin(theta)

    dist_sq = (x_c - x0)**2 + (y_c - y0)**2
    weights = np.exp(-dist_sq / (2 * sigma**2))
    weights /= np.sum(weights)

    y_mean = np.array([np.sum(weights * x_c), np.sum(weights * y_c)])
    score_raw = (y_mean - np.array([x0, y0])) / (sigma**2)
    score_mag = np.linalg.norm(score_raw)
    score_unit = score_raw / score_mag if score_mag > 1e-9 else score_raw

    E_yyT = np.array([[np.sum(weights * x_c**2), np.sum(weights * x_c * y_c)],
                      [np.sum(weights * x_c * y_c), np.sum(weights * y_c**2)]])
    cov = E_yyT - np.outer(y_mean, y_mean)
    jacobian = (1 / sigma**4) * cov - (1 / sigma**2) * np.eye(2)
    eigenvalues, eigenvectors = np.linalg.eigh(jacobian)

    return score_raw, score_unit, eigenvectors, eigenvalues


state = {'x': 0.25, 'y': 1.2}

a_slider     = widgets.IntSlider(value=8,    min=2,    max=16,  step=1,    description='a')
sigma_slider = widgets.FloatSlider(value=0.2, min=0.01, max=0.7,  step=0.01,             description='σ')
phase_slider = widgets.FloatSlider(value=0.0,  min=0.0,  max=2*np.pi, step=np.pi/100, description='φ')

fig, ax = plt.subplots(figsize=(9, 9))
fig.canvas.toolbar_visible = False

def inv_max_curvature_curve(x, y):
    """Finite-difference curvature for a closed parametric curve."""
    dx  = np.gradient(x);  dy  = np.gradient(y)
    ddx = np.gradient(dx); ddy = np.gradient(dy)
    kappa = np.abs(dx * ddy - dy * ddx) / (dx**2 + dy**2)**1.5
    return 1.0 / np.max(kappa)

def inv_max_curvature_polar(a, phase, num_theta=5000):
    """Exact curvature via polar formula: κ = |r²+2r'²−rr''| / (r²+r'²)^(3/2)."""
    theta = np.linspace(0, 2 * np.pi, num_theta)
    r   =  1 + (1.0 / a) * np.sin(a * theta + phase)
    dr  =  np.cos(a * theta + phase)
    ddr = -a * np.sin(a * theta + phase)
    kappa = np.abs(r**2 + 2*dr**2 - r*ddr) / (r**2 + dr**2)**1.5
    return 1.0 / np.max(kappa)

def redraw():
    a     = a_slider.value
    sigma = sigma_slider.value
    phase = phase_slider.value
    x0, y0 = state['x'], state['y']

    X, Y, exact_density, x_true, y_true, x_mode, y_mode = compute_exact_density_and_mode(a, sigma, phase=phase, num_rays=2000)
    score, score_unit, eigvecs, eigvals = compute_local_properties(x0, y0, a, sigma, phase=phase)

    for i in range(2):
        if np.dot(eigvecs[:, i], score_unit) < 0:
            eigvecs[:, i] *= -1

    ax.cla()
    ax.pcolormesh(X, Y, exact_density, shading='auto', cmap='viridis', alpha=0.5)
    ax.plot(x_true, y_true, color='black', linewidth=1, label='True curve $C$')
    ax.plot(np.append(x_mode, x_mode[0]), np.append(y_mode, y_mode[0]),
            color='red', linewidth=2, label='Mode curve $M$')

    L = 0.3

    def unit_toward(tx, ty):
        v = np.array([tx - x0, ty - y0])
        n = np.linalg.norm(v)
        return v / n if n > 1e-9 else v

    idx_true = np.argmin((x_true - x0)**2 + (y_true - y0)**2)
    d_true = unit_toward(x_true[idx_true], y_true[idx_true])
    ax.annotate("", xy=(x0 + L * d_true[0], y0 + L * d_true[1]), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='black', alpha=0.5, lw=2), zorder=5)

    idx_mode = np.argmin((x_mode - x0)**2 + (y_mode - y0)**2)
    d_mode = unit_toward(x_mode[idx_mode], y_mode[idx_mode])
    ax.annotate("", xy=(x0 + L * d_mode[0], y0 + L * d_mode[1]), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='red', alpha=0.5, lw=2), zorder=5)

    ax.quiver(x0, y0, score_unit[0], score_unit[1], color='white', alpha=1,
              scale=1/L, scale_units='xy', label='Score Dir', zorder=3)
    ax.quiver(x0, y0, eigvecs[0, 0], eigvecs[1, 0], color='cyan', alpha=0.5,
              scale=1/L, scale_units='xy', label='Eig1 Dir', zorder=4)
    ax.quiver(x0, y0, eigvecs[0, 1], eigvecs[1, 1], color='magenta', alpha=0.5,
              scale=1/L, scale_units='xy', label='Eig2 Dir', zorder=4)

    roc_true = inv_max_curvature_polar(a, phase)
    roc_mode = inv_max_curvature_curve(x_mode, y_mode)

    ax.set_aspect('equal')
    ax.set_title(
        f"Score Mag: {np.linalg.norm(score):.2f} | λ₁: {eigvals[0]:.2f}, λ₂: {eigvals[1]:.2f}\n"
        f"1/κ_max — true: {roc_true:.3f}  mode: {roc_mode:.3f}"
    )
    ax.legend(loc='upper right', frameon=True)
    fig.canvas.draw_idle()


def on_click(event):
    if event.inaxes == ax:
        state['x'], state['y'] = event.xdata, event.ydata
        redraw()

fig.canvas.mpl_connect('button_press_event', on_click)
a_slider.observe(lambda _: redraw(), names='value')
sigma_slider.observe(lambda _: redraw(), names='value')

phase_slider.observe(lambda _: redraw(), names='value')
display(VBox([HBox([a_slider, sigma_slider, phase_slider]), fig.canvas]))
redraw()

In [6]:
import numpy as np
from scipy.ndimage import gaussian_filter1d


# ── helpers ────────────────────────────────────────────────────────────────────

def _wiggly_circle(a, phase, num_theta=1000):
    """Points, arc-length element, and derivatives for r = 1 + sin(aθ+φ)/a."""
    theta = np.linspace(0, 2 * np.pi, num_theta)
    r   = 1 + (1.0 / a) * np.sin(a * theta + phase)
    dr  = np.cos(a * theta + phase)
    x_c = r * np.cos(theta)
    y_c = r * np.sin(theta)
    ds  = np.sqrt(r**2 + dr**2)
    return theta, x_c, y_c, ds


def _compute_mode_curve(a, sigma, phase, num_rays=2000, grid_res=500, num_theta=200):
    """Mode curve via radial ray search with sub-pixel refinement + smoothing."""
    theta, x_c, y_c, ds = _wiggly_circle(a, phase, num_theta)

    limit  = 2
    phi    = np.linspace(0, 2 * np.pi, num_rays)
    R_span = np.linspace(0, limit * 0.9, grid_res)
    PHI, R_grid = np.meshgrid(phi, R_span)
    X_ray = R_grid * np.cos(PHI)
    Y_ray = R_grid * np.sin(PHI)

    density_ray = np.zeros((grid_res, num_rays))
    for i in range(grid_res):
        sq = (X_ray[i, :, np.newaxis] - x_c)**2 + (Y_ray[i, :, np.newaxis] - y_c)**2
        density_ray[i] = np.trapezoid(np.exp(-sq / (2 * sigma**2)) * ds, x=theta, axis=-1)

    idx_c = np.clip(np.argmax(density_ray, axis=0), 1, grid_res - 2)
    j  = np.arange(num_rays)
    f0, f1, f2 = density_ray[idx_c - 1, j], density_ray[idx_c, j], density_ray[idx_c + 1, j]
    denom  = f0 - 2 * f1 + f2
    offset = np.where(np.abs(denom) > 1e-30, 0.5 * (f0 - f2) / denom, 0.0)
    R_mode = R_span[idx_c] + offset * (R_span[1] - R_span[0])
    R_mode = gaussian_filter1d(R_mode, sigma=num_rays / 200, mode='wrap')

    return R_mode * np.cos(phi), R_mode * np.sin(phi)


def _score_and_jacobian(x0, y0, x_c, y_c, sigma):
    """Score, eigenvectors (unit), and eigenvalues of J = ∇²log p_σ at (x0,y0)."""
    dist_sq = (x_c - x0)**2 + (y_c - y0)**2
    w = np.exp(-dist_sq / (2 * sigma**2))
    w /= w.sum()

    mu = np.array([w @ x_c, w @ y_c])
    score = (mu - np.array([x0, y0])) / sigma**2

    E_yyT = np.array([[w @ (x_c**2),    w @ (x_c * y_c)],
                      [w @ (x_c * y_c), w @ (y_c**2)   ]])
    cov = E_yyT - np.outer(mu, mu)
    J   = cov / sigma**4 - np.eye(2) / sigma**2

    eigvals, eigvecs = np.linalg.eigh(J)
    return score, eigvecs, eigvals          # eigvecs: columns are unit eigenvectors


def _project(x0, y0, x_c, y_c):
    """Closest point on a discrete curve to (x0, y0)."""
    idx = np.argmin((x_c - x0)**2 + (y_c - y0)**2)
    return np.array([x_c[idx], y_c[idx]])


def _curvature_polar(a, phase, num_theta=5000):
    """Exact curvature stats for wiggly circle via κ = |r²+2r'²−rr''|/(r²+r'²)^(3/2)."""
    theta = np.linspace(0, 2 * np.pi, num_theta)
    r   =  1 + (1.0 / a) * np.sin(a * theta + phase)
    dr  =  np.cos(a * theta + phase)
    ddr = -a * np.sin(a * theta + phase)
    kappa = np.abs(r**2 + 2 * dr**2 - r * ddr) / (r**2 + dr**2)**1.5
    return kappa.max(), kappa.mean()


def _curvature_discrete(x, y):
    """Curvature stats for a closed discrete curve via finite differences."""
    dx  = np.gradient(x);  dy  = np.gradient(y)
    ddx = np.gradient(dx); ddy = np.gradient(dy)
    kappa = np.abs(dx * ddy - dy * ddx) / (dx**2 + dy**2)**1.5
    return kappa.max(), kappa.mean()


# ── main API ───────────────────────────────────────────────────────────────────

def get_properties(x0, y0, a, sigma, phase=0.0):
    """
    Returns a dict of local geometric properties at point (x0, y0).

    Keys
    ----
    score          : (2,)   ∇ log p_σ(x)
    eigvecs        : (2,2)  unit eigenvectors of J(x), columns ordered by eigenvalue
    eigvals        : (2,)   eigenvalues of J(x)  (≤ 0)
    eigvecs_scaled : (2,2)  eigvecs scaled by |eigval|  (unnormalized)
    proj_true      : (2,)   π₀(x) — closest point on C
    proj_mode      : (2,)   π_σ(x) — closest point on M_σ
    kappa_max_true : float  max curvature of C  (exact)
    kappa_mean_true: float  mean curvature of C (exact)
    kappa_max_mode : float  max curvature of M_σ
    kappa_mean_mode: float  mean curvature of M_σ
    """
    _, x_c, y_c, _   = _wiggly_circle(a, phase)
    x_mode, y_mode   = _compute_mode_curve(a, sigma, phase)

    score, eigvecs, eigvals = _score_and_jacobian(x0, y0, x_c, y_c, sigma)
    eigvecs_scaled = eigvecs * np.abs(eigvals)

    proj_true = _project(x0, y0, x_c, y_c)
    proj_mode = _project(x0, y0, x_mode, y_mode)

    kappa_max_true,  kappa_mean_true  = _curvature_polar(a, phase)
    kappa_max_mode,  kappa_mean_mode  = _curvature_discrete(x_mode, y_mode)

    return dict(
        score=score,
        eigvecs=eigvecs,
        eigvals=eigvals,
        eigvecs_scaled=eigvecs_scaled,
        proj_true=proj_true,
        proj_mode=proj_mode,
        kappa_max_true=kappa_max_true,
        kappa_mean_true=kappa_mean_true,
        kappa_max_mode=kappa_max_mode,
        kappa_mean_mode=kappa_mean_mode,
    )


# ── quick smoke-test ───────────────────────────────────────────────────────────
props = get_properties(0.5, 0.5, a=8, sigma=0.1, phase=0.0)
for k, v in props.items():
    print(f"{k:20s}: {v}")

score               : [21.77738742  4.58650421]
eigvecs             : [[-0.97044897  0.24130644]
 [-0.24130644 -0.97044897]]
eigvals             : [-96.9714877  -49.25914463]
eigvecs_scaled      : [[-94.10588033  11.88654876]
 [-23.39984434 -47.80348615]]
proj_true           : [0.70854245 0.53619927]
proj_mode           : [0.72459854 0.56977474]
kappa_max_true      : 9.306119698860403
kappa_mean_true     : 3.7268371400735645
kappa_max_mode      : 5.380090425367281
kappa_mean_mode     : 3.1700183103665673


In [4]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import os

os.makedirs("plots", exist_ok=True)

# Initial values matching slider defaults
a, sigma, x0, y0 = 8, 0.2, 0.25, 1.2
n_frames = 120
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

fig, ax = plt.subplots(figsize=(9, 9))
plt.close(fig)  # prevent inline display

def render_frame(phase):
    ax.cla()
    X, Y, exact_density, x_true, y_true, x_mode, y_mode = compute_exact_density_and_mode(
        a, sigma, phase=phase, grid_res=300)
    score, score_unit, eigvecs, eigvals = compute_local_properties(x0, y0, a, sigma, phase=phase)

    for i in range(2):
        if np.dot(eigvecs[:, i], score_unit) < 0:
            eigvecs[:, i] *= -1

    ax.pcolormesh(X, Y, exact_density, shading='auto', cmap='viridis', alpha=0.5)
    ax.plot(x_true, y_true, color='black', linewidth=1, label='True curve $C$')
    ax.plot(np.append(x_mode, x_mode[0]), np.append(y_mode, y_mode[0]),
            color='red', linewidth=2, label='Mode curve $M$')

    L = 0.3

    def unit_toward(tx, ty):
        v = np.array([tx - x0, ty - y0])
        n = np.linalg.norm(v)
        return v / n if n > 1e-9 else v

    idx_true = np.argmin((x_true - x0)**2 + (y_true - y0)**2)
    d_true = unit_toward(x_true[idx_true], y_true[idx_true])
    ax.annotate("", xy=(x0 + L * d_true[0], y0 + L * d_true[1]), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='black', alpha=0.5, lw=2), zorder=4)

    idx_mode = np.argmin((x_mode - x0)**2 + (y_mode - y0)**2)
    d_mode = unit_toward(x_mode[idx_mode], y_mode[idx_mode])
    ax.annotate("", xy=(x0 + L * d_mode[0], y0 + L * d_mode[1]), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='red', alpha=0.5, lw=2), zorder=4)

    ax.quiver(x0, y0, score_unit[0], score_unit[1], color='white', alpha=0.5,
              scale=1/L, scale_units='xy', label='Score Dir', zorder=5)
    ax.quiver(x0, y0, eigvecs[0, 0], eigvecs[1, 0], color='cyan', alpha=0.5,
              scale=1/L, scale_units='xy', label='Eig1 Dir', zorder=5)
    ax.quiver(x0, y0, eigvecs[0, 1], eigvecs[1, 1], color='magenta', alpha=0.5,
              scale=1/L, scale_units='xy', label='Eig2 Dir', zorder=5)

    ax.set_aspect('equal')
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_title(f"φ = {phase:.2f}  |  Score Mag: {np.linalg.norm(score):.2f}"
                 f"  |  λ₁: {eigvals[0]:.2f}, λ₂: {eigvals[1]:.2f}")
    ax.legend(loc='upper right', frameon=True)

ani = animation.FuncAnimation(fig, lambda i: render_frame(phases[i]),
                               frames=n_frames, interval=100)
ani.save("plots/09-phase-sweep.gif", writer='pillow', fps=15)
print("Saved plots/09-phase-sweep.gif")

Saved plots/09-phase-sweep.gif


# Dataset of score behaviour

### Summary: Local Geometry at a Point $x$

Let $\mathcal{C} \subset \mathbb{R}^d$ be the true data manifold and $\mathcal{M}_\sigma \subset \mathbb{R}^d$ the mode manifold of $p_\sigma$.

**Score.**
$$s(x) = \nabla_x \log p_\sigma(x) = \frac{\mathbb{E}_{y \sim p_\sigma(y|x)}[y] - x}{\sigma^2} \in \mathbb{R}^d$$

**Jacobian of the score (Hessian of log-density).**
$$J(x) = \nabla_x s(x) = \nabla_x^2 \log p_\sigma(x) = \frac{1}{\sigma^4}\operatorname{Cov}_{p_\sigma(y|x)}(y) - \frac{1}{\sigma^2}I_d$$

$J(x)$ is symmetric negative semi-definite. Its eigen-decomposition $J(x) = \sum_i \lambda_i v_i v_i^\top$ yields:
- **Eigenvectors** $v_i(x) \in \mathbb{R}^d$: principal curvature directions of $\log p_\sigma$ at $x$.
- **Eigenvalues** $\lambda_i(x) \leq 0$: the most negative eigenvalue corresponds to the direction of steepest log-density curvature (normal to the manifold); the least negative corresponds to the tangential direction.

**Projection onto the true manifold.**
$$\pi_0(x) = \operatorname{argmin}_{z \in \mathcal{C}} \|x - z\|$$

The vector $x - \pi_0(x)$ lies in the normal space of $\mathcal{C}$ at $\pi_0(x)$ (when $x$ is within the reach of $\mathcal{C}$).

**Projection onto the mode manifold.**
$$\pi_\sigma(x) = \operatorname{argmin}_{z \in \mathcal{M}_\sigma} \|x - z\|$$

$\mathcal{M}_\sigma$ is the set of local maxima of $p_\sigma$ along every normal direction; equivalently, $s(z) = 0$ on $\mathcal{M}_\sigma$. As $\sigma \to 0$, $\mathcal{M}_\sigma \to \mathcal{C}$ and $\pi_\sigma(x) \to \pi_0(x)$.